# Hotfish: temperature vs developmental stage

Refined reproduction of the key staging figures from `results/nlammers/20260528`.

Three analyses:
1. **Stage vs temperature** for the 36 hpf collection, built up cohort-by-cohort (25–32C morph → add 34+35C → add transcriptional stage).
2. **Transcriptional stage vs morphological stage** scatter — 36 hpf only, then all collection times — with per-cohort SD.
3. **Stage variability (SD) vs temperature** — morph first, then morph + transcription.

### Where the two stage estimates come from (audit)
- **Morphological stage** (`mdl_stage_hpf`): upstream sklearn `PolynomialFeatures → LinearRegression` model (`morph_stage_model.joblib`) applied to morph-VAE PCA coordinates. Precomputed in the cached hotfish tables; **not** re-fit here. A separate spline-nearest-neighbor stage (`spline_stage_hpf`) exists but is not what these figures use.
- **Transcriptional stage** (`pseudostage`): upstream Hooke regression (`bead_expt_linear` → `time_predictions.csv`), per-embryo nn-transcriptional age, merged onto morphology via `morphseq_metadata.csv`. **Not** re-fit here.

See `hotfish_stage_utils.py` for full provenance notes.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import hotfish_stage_utils as hs

# ---- 19C toggle ---------------------------------------------------------- #
# Default FALSE (matches the 20260528 pass). Set True to include the 19C
# cohort everywhere; figures then write to figures/with_19C/ instead of
# figures/no_19C/ so the two versions never overwrite each other.
INCLUDE_19C = False
hs.set_include_19c(INCLUDE_19C)

hs.set_light_style()
print('data cache :', hs.CACHE_DIR)
print('figure dir :', hs.fig_dir())
print('include 19C:', hs.INCLUDE_19C)

## Load data

Prefers the pre-built `joint_141_morph_seq.csv` (both stages joined). If that isn't present, assembles the joint table from `hf_pca_morph_df.csv` (morph, always available) plus a seq-staging source. The morph-distance table is loaded separately for the morphological-noise variant of analysis (3).

In [ ]:
def load_joint():
    """Return the matched-embryo table with morph + (if available) seq stage."""
    cached = hs.CACHE_DIR / 'joint_141_morph_seq.csv'
    if cached.exists():
        joint = pd.read_csv(cached)
        print(f'loaded cached joint table: {cached} ({len(joint)} rows)')
        return joint

    # Fallback: assemble from the morph table (always on-disk) + seq staging.
    morph = pd.read_csv(hs.CACHE_DIR / 'hf_pca_morph_df.csv')
    print(f'no cached joint table; assembling from hf_pca_morph_df.csv ({len(morph)} rows)')
    seq = None
    for cand in [hs.CACHE_DIR / 'seq_to_morph_pca_pd.csv',
                 hs.CACHE_DIR / 'time_predictions.csv']:
        if cand.exists():
            seq = pd.read_csv(cand)
            print(f'  + seq staging from {cand.name}')
            break
    if seq is not None and hs.SEQ_STAGE_COL in seq.columns:
        key = 'snip_id' if 'snip_id' in seq.columns else ('sample' if 'sample' in seq.columns else None)
        if key and key in morph.columns:
            morph = morph.merge(seq[[key, hs.SEQ_STAGE_COL]].drop_duplicates(key), on=key, how='left')
    if hs.SEQ_STAGE_COL not in morph.columns:
        print('  NOTE: no transcriptional stage source found — seq (pseudostage) panels will be skipped.')
    return morph


joint = hs.drop_cold(load_joint())
HAS_SEQ = hs.SEQ_STAGE_COL in joint.columns and joint[hs.SEQ_STAGE_COL].notna().any()

# Morph-distance table (for the morphological-noise variant of analysis 3).
morph_dist_path = hs.CACHE_DIR / 'hf_pca_morph_df_with_spline_distance.csv'
hf_morph = hs.drop_cold(pd.read_csv(morph_dist_path)) if morph_dist_path.exists() else None

# ---- Arrhenius fit (true model, per hf2_seq_morph_arrhenius.ipynb) -------- #
# stage = A * t * exp(-E/(R*T_kelvin)); BOTH A and E fit by least_squares to the
# 25/28.5/32C cohorts (no 19C). E is a FREE parameter, not pinned to a literature value.
ARR_MORPH = hs.fit_arrhenius(joint, hs.MORPH_STAGE_COL)          # (A, E)
ARR_SEQ = hs.fit_arrhenius(joint, hs.SEQ_STAGE_COL) if HAS_SEQ else (np.nan, np.nan)
print(f'Arrhenius fit temps      : {hs.ARRHENIUS_FIT_TEMPS}')
print(f'Arrhenius (morph): A={ARR_MORPH[0]:.4g}  E={ARR_MORPH[1]:.4g} J/mol')
if HAS_SEQ:
    print(f'Arrhenius (seq)  : A={ARR_SEQ[0]:.4g}  E={ARR_SEQ[1]:.4g} J/mol')

# ---- 28.5C standardization offset ---------------------------------------- #
# Mean (seq - morph) stage at the 28.5C control; subtracting it from the
# transcriptional stage puts the control cohort on the y=x line so the two
# staging axes are directly comparable.
SEQ_OFFSET = hs.seq_morph_offset(joint) if HAS_SEQ else 0.0
if HAS_SEQ:
    print(f'seq-morph offset at {hs.OFFSET_REF_TEMP}C: {SEQ_OFFSET:+.3f} hpf '
          '(subtracted from transcriptional stage)')

print(f'joint rows: {len(joint)}  |  temperatures: {sorted(joint["temperature"].unique())}')
print(f'transcriptional stage available: {HAS_SEQ}')
joint.head()

## (i) Stage vs temperature — 36 hpf collection, built up in stages

Cohort-mean stage (with bootstrap SE) against temperature, for the 36 hpf collection only. Panels are added in the requested order:
1. morphological stage, 25–32C only;
2. add the 33.5+35C cohorts;
3. overlay transcriptional stage (○ morph, ◇ transcription, standardized by the 28.5C offset).

The dashed grey curve is the **Arrhenius** expected stage at 36 hpf: `A·t·exp(-E/RT)`, with **both A and E fit by least-squares** to the **25/28.5/32C** cohorts (source: `hf2_seq_morph_arrhenius.ipynb`, results/nlammers/20250319; E is a free parameter, not pinned to the Toulany 2023 literature value). *Not* the linear approximation from the 20260504 notebooks.

In [ ]:
TARGET_TP = 36.0

# ordered temperature build-up for the 36hpf series
STEP1_TEMPS = [25.0, 28.5, 32.0]                # 25–32C
STEP2_TEMPS = STEP1_TEMPS + [33.5, 35.0]        # add the hot cohorts
if hs.INCLUDE_19C:
    STEP1_TEMPS = [19.0] + STEP1_TEMPS
    STEP2_TEMPS = [19.0] + STEP2_TEMPS

tp36 = joint.loc[np.isclose(joint['timepoint'].astype(float), TARGET_TP)].copy()
summ36 = hs.cohort_stage_summary(tp36)

# marker convention: morph = circle, transcription = diamond
MORPH_MARKER = 'o'
SEQ_MARKER = 'D'


def _stage_vs_temp(ax, summ, stage_col, temps, arr_params, *, label):
    """Cohort-mean morphological stage vs temperature (○, colored by
    temperature), with the fitted Arrhenius reference curve. ``arr_params`` is
    the (A, E) tuple from hs.fit_arrhenius."""
    d = summ.loc[summ['temperature'].isin(temps)].sort_values('temperature')
    mean_c, se_c = f'{stage_col}_mean', f'{stage_col}_mean_boot_se'
    ax.errorbar(d['temperature'], d[mean_c], yerr=d[se_c], fmt='none',
                ecolor='#666666', elinewidth=0.9, capsize=2.5, alpha=0.8, zorder=1)
    hs.temperature_scatter_fixed_marker(ax, d['temperature'], d[mean_c],
                                        d['temperature'], marker=MORPH_MARKER,
                                        s=90, zorder=2)
    # fitted Arrhenius reference at 36 hpf across the shown temperature range
    tvec = np.linspace(min(temps), max(temps), 100)
    ax.plot(tvec, hs.arrhenius_expected_stage(TARGET_TP, tvec, arr_params), '--',
            color='#999999', lw=1.2, zorder=0, label='Arrhenius fit')
    ax.set_xlabel('temperature (C)')
    ax.set_ylabel(label)


# --- step 1: morph, 25-32C ---
fig, ax = plt.subplots(figsize=(5.4, 4.4))
_stage_vs_temp(ax, summ36, hs.MORPH_STAGE_COL, STEP1_TEMPS, ARR_MORPH, label=hs.MORPH_STAGE_LABEL)
ax.set_title('36 hpf · morphological stage · 25–32C')
ax.legend(frameon=False, fontsize=8)
hs.savefig(fig, '01i_a_stage_vs_temp_morph_25_32')
plt.show()

In [ ]:
# --- step 2: morph, add 33.5+35C ---
fig, ax = plt.subplots(figsize=(5.4, 4.4))
_stage_vs_temp(ax, summ36, hs.MORPH_STAGE_COL, STEP2_TEMPS, ARR_MORPH, label=hs.MORPH_STAGE_LABEL)
ax.set_title('36 hpf · morphological stage · all temperatures')
ax.legend(frameon=False, fontsize=8)
hs.savefig(fig, '01i_b_stage_vs_temp_morph_all')
plt.show()

In [ ]:
# --- step 3: overlay transcriptional stage (○ morph, ◇ transcription) ---
if HAS_SEQ:
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    _stage_vs_temp(ax, summ36, hs.MORPH_STAGE_COL, STEP2_TEMPS, ARR_MORPH,
                   label='inferred stage (hpf)')
    d = summ36.loc[summ36['temperature'].isin(STEP2_TEMPS)].sort_values('temperature')
    mean_c, se_c = f'{hs.SEQ_STAGE_COL}_mean', f'{hs.SEQ_STAGE_COL}_mean_boot_se'
    seq_mean = d[mean_c] - SEQ_OFFSET   # standardized to the 28.5C control
    ax.errorbar(d['temperature'], seq_mean, yerr=d[se_c], fmt='none',
                ecolor='#666666', elinewidth=0.9, capsize=2.5, alpha=0.8, zorder=1)
    # transcriptional markers keep the temperature color; only the SYMBOL differs
    hs.temperature_scatter_fixed_marker(ax, d['temperature'], seq_mean,
                                        d['temperature'], marker=SEQ_MARKER, s=80,
                                        facecolor_alpha=0.55, zorder=3)
    ax.set_title('36 hpf · morphological (○) + transcriptional (◇) stage')
    # legend: dashed Arrhenius line + a neutral symbol key
    line_handles, _ = ax.get_legend_handles_labels()
    from matplotlib.lines import Line2D
    sym_handles = [
        Line2D([0], [0], marker=MORPH_MARKER, linestyle='none', markerfacecolor='#cccccc',
               markeredgecolor='#333333', markersize=8, label='morphological'),
        Line2D([0], [0], marker=SEQ_MARKER, linestyle='none', markerfacecolor='#cccccc',
               markeredgecolor='#333333', markersize=7, label='transcriptional'),
    ]
    ax.legend(handles=line_handles + sym_handles, frameon=False, fontsize=8, loc='best')
    hs.savefig(fig, '01i_c_stage_vs_temp_morph_and_seq')
    plt.show()
else:
    print('transcriptional stage unavailable — skipping step 3.')

## (ii) Transcriptional stage vs morphological stage

Per-embryo scatter, colored by temperature and marker-coded by collection time. Cohort SD is shown as error bars on the cohort means (larger points). First 36 hpf only, then all collection times.

In [ ]:
def seq_vs_morph_scatter(df, title, name):
    """Transcriptional vs morphological stage, both temperature-colored and
    marker-coded by collection time. The transcriptional axis is standardized
    by the 28.5C offset so the control cohort sits on y=x."""
    df = df.copy()
    df['_seq_std'] = df[hs.SEQ_STAGE_COL] - SEQ_OFFSET   # 28.5C standardization
    summ = hs.cohort_stage_summary(df, stage_cols=(hs.MORPH_STAGE_COL, '_seq_std'))

    fig, ax = plt.subplots(figsize=(5.4, 4.8))
    # per-embryo points (faint background layer, same color/marker convention)
    hs.temperature_timepoint_scatter(ax, df[hs.MORPH_STAGE_COL], df['_seq_std'],
                                     df['temperature'], df['timepoint'],
                                     s=26, alpha=0.35, linewidth=0.0, zorder=1)
    # cohort means with SD error bars in BOTH axes
    mx, sx = f'{hs.MORPH_STAGE_COL}_mean', f'{hs.MORPH_STAGE_COL}_std'
    my, sy = '_seq_std_mean', '_seq_std_std'
    ax.errorbar(summ[mx], summ[my], xerr=summ[sx], yerr=summ[sy], fmt='none',
                ecolor='#444444', elinewidth=0.9, capsize=2.5, alpha=0.8, zorder=2)
    hs.temperature_timepoint_scatter(ax, summ[mx], summ[my], summ['temperature'],
                                     summ['timepoint'], s=95, alpha=0.95,
                                     linewidth=0.6, add_legend=True, zorder=3)
    hs.add_identity(ax, df[hs.MORPH_STAGE_COL], df['_seq_std'])
    ax.set_xlabel(hs.MORPH_STAGE_LABEL)
    ax.set_ylabel(f'{hs.SEQ_STAGE_LABEL}\n(standardized to 28.5C)')
    ax.set_title(title)
    hs.savefig(fig, name)
    plt.show()
    return summ


if HAS_SEQ:
    _ = seq_vs_morph_scatter(tp36, f'Transcriptional vs morphological stage · 36 hpf (n={len(tp36)})',
                             '02ii_a_seq_vs_morph_36hpf')
    _ = seq_vs_morph_scatter(joint, f'Transcriptional vs morphological stage · all collections (n={len(joint)})',
                             '02ii_b_seq_vs_morph_all')
else:
    print('transcriptional stage unavailable — skipping analysis (ii).')

## (iii) Stage variability (SD) vs temperature

Per-temperature stage variability = mean over collection timepoints of the within-cohort SD, with bootstrap SE. Morphological stage first, then morph + transcription overlaid.

In [ ]:
# --- morph only ---
morph_var = hs.timepoint_average_variability_bootstrap(joint, hs.MORPH_STAGE_COL)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
ax.errorbar(morph_var['temperature'], morph_var['variability_mean'],
            yerr=morph_var['variability_boot_se'], fmt='o-', color='#b2182b',
            ecolor='#b2182b', elinewidth=1.0, capsize=3, markersize=6, label='morphological')
ax.set_xlabel('temperature (C)')
ax.set_ylabel('stage variability (mean cohort SD, hpf)')
ax.set_title('Stage variability vs temperature · morphological')
ax.legend(frameon=False, fontsize=9)
hs.savefig(fig, '03iii_a_variability_vs_temp_morph')
plt.show()
morph_var

In [ ]:
# --- morph + transcription ---
if HAS_SEQ:
    seq_var = hs.timepoint_average_variability_bootstrap(joint, hs.SEQ_STAGE_COL)
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    ax.errorbar(morph_var['temperature'], morph_var['variability_mean'],
                yerr=morph_var['variability_boot_se'], fmt='o-', color='#b2182b',
                ecolor='#b2182b', elinewidth=1.0, capsize=3, markersize=6, label='morphological')
    ax.errorbar(seq_var['temperature'], seq_var['variability_mean'],
                yerr=seq_var['variability_boot_se'], fmt='D--', color='#2166ac',
                ecolor='#2166ac', elinewidth=1.0, capsize=3, markersize=6, label='transcriptional')
    ax.set_xlabel('temperature (C)')
    ax.set_ylabel('stage variability (mean cohort SD, hpf)')
    ax.set_title('Stage variability vs temperature · morph + transcription')
    ax.legend(frameon=False, fontsize=9)
    hs.savefig(fig, '03iii_b_variability_vs_temp_morph_and_seq')
    plt.show()
    seq_var
else:
    print('transcriptional stage unavailable — skipping the morph+seq variability plot.')

### Optional: morphological *noise* variant of (iii)

The 20260528 pass also expressed morphological variability as distance-from-WT-spline SD (`morph_dist_spline`) rather than stage SD. Shown here for completeness if the distance table is present.

In [ ]:
if hf_morph is not None and hs.MORPH_DIST_COL in hf_morph.columns:
    dist_var = hs.timepoint_average_variability_bootstrap(hf_morph, hs.MORPH_DIST_COL)
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    ax.errorbar(dist_var['temperature'], dist_var['variability_mean'],
                yerr=dist_var['variability_boot_se'], fmt='o-', color='#333333',
                elinewidth=1.0, capsize=3, markersize=6)
    ax.set_xlabel('temperature (C)')
    ax.set_ylabel('morph distance-from-spline variability (SD)')
    ax.set_title('Morphological noise vs temperature')
    hs.savefig(fig, '03iii_c_morph_distance_variability_vs_temp')
    plt.show()
    dist_var
else:
    print('morph-distance table not present — skipping the distance-based variant.')